<a href="https://colab.research.google.com/github/Maria-lin/F1-Analytics/blob/main/detection_anomalies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏧 Détection des Comportements Atypiques des GAB — v2
## Analyse enrichie : clustering, géographie, consolidation annuelle et justification

---
> **Nouveautés v2 :** clustering KMeans · contexte géographique · consolidation annuelle · justification automatique · suppression doublons  
> **Données :** fiche_identite_gab_mensuel + table_ref_gab_finale (coords corrigées BAN)

---

## 1. ⚙️ Imports et Configuration

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

try:
    import shap
    SHAP_OK = True
except ImportError:
    SHAP_OK = False

try:
    from sklearn.tree import DecisionTreeClassifier, export_text
    TREE_OK = True
except ImportError:
    TREE_OK = False

sns.set_theme(style='whitegrid', font_scale=1.1)
COULEUR_NORMAL   = '#2196F3'
COULEUR_ANOMALIE = '#F44336'
COULEUR_ACCENT   = '#FF9800'
PALETTE_CLUSTERS = ['#1976D2','#388E3C','#F57C00','#7B1FA2','#D32F2F','#00796B']

print('✅ Librairies chargées.')

## 2. 📂 Chargement des Données

> **En production Dataiku**, remplacer la simulation par :
> ```python
> import dataiku
> df = dataiku.Dataset('fiche_identite_gab_mensuel').get_dataframe()
> ```

In [ ]:
np.random.seed(42)
n_normal, n_anom = 1800, 120   # ~150 GAB × 12 mois

def generer_dataset():
    types_gab  = np.random.choice(['Interne','Externe'], n_normal+n_anom, p=[0.35,0.65])
    codes_post = np.random.choice(['75001','69001','13001','33000','31000',
                                   '59000','44000','67000','06000','34000',
                                   '01000','05000','48000','23000'], n_normal+n_anom)
    # Contexte géo : grande ville vs rural
    grande_ville = np.isin(codes_post, ['75001','69001','13001','33000','59000','06000'])

    df_n = pd.DataFrame({
        'num_automate'      : [f'GAB_{i:04d}' for i in np.random.randint(0,150,n_normal)],
        'annee'             : np.random.choice([2024,2025], n_normal),
        'mois'              : np.random.randint(1,13, n_normal),
        'type_gab_e_i'      : types_gab[:n_normal],
        'code_postal'       : codes_post[:n_normal],
        'longitude'         : np.random.uniform(-4.5,8.2, n_normal),
        'latitude'          : np.random.uniform(42.3,51.1, n_normal),
        'ret_nb'            : np.where(grande_ville[:n_normal],
                                np.random.normal(1100,200,n_normal),
                                np.random.normal(500,120,n_normal)).clip(50),
        'ret_montant_moyen' : np.random.normal(150,30,n_normal).clip(50),
        'ret_montant_total' : np.random.normal(120000,25000,n_normal).clip(5000),
        'ret_montant_stddev': np.random.normal(60,15,n_normal).clip(5),
        'ret_nb_nuit'       : np.random.normal(80,25,n_normal).clip(0),
        'ret_nb_weekend'    : np.random.normal(180,40,n_normal).clip(0),
        'ret_pct_nuit'      : np.random.normal(10,3,n_normal).clip(0,100),
        'ret_pct_weekend'   : np.random.normal(22,5,n_normal).clip(0,100),
        'cap_nb'            : np.random.poisson(4,n_normal),
        'taux_capture_pct'  : np.random.normal(0.5,0.3,n_normal).clip(0),
        'nb_jours_actifs'   : np.random.randint(15,30,n_normal),
        'nb_ope_reseau_cb': np.random.normal(350,60,n_normal).clip(0),
        'nb_ope_reseau_visa': np.random.normal(180,40,n_normal).clip(0),
        'nb_ope_reseau_mastercard': np.random.normal(120,30,n_normal).clip(0),
        'nb_ope_reseau_interne': np.random.normal(80,20,n_normal).clip(0),
        'nb_ope_reseau_franfinance': np.random.normal(30,10,n_normal).clip(0),
        'nb_ope_reseau_accord': np.random.normal(25,10,n_normal).clip(0),
        'nb_ope_reseau_trionis': np.random.normal(15,8,n_normal).clip(0),
        'nb_ope_reseau_ppl': np.random.normal(12,6,n_normal).clip(0),
        'nb_ope_reseau_casino': np.random.normal(10,5,n_normal).clip(0),
        'nb_ope_reseau_configona': np.random.normal(8,4,n_normal).clip(0),
        'nb_ope_reseau_cos': np.random.normal(6,3,n_normal).clip(0),
        'nb_ope_reseau_jcb': np.random.normal(4,3,n_normal).clip(0),
        'nb_ope_reseau_postepargne': np.random.normal(5,3,n_normal).clip(0),
        'nb_ope_reseau_diners_et_discovery': np.random.normal(3,2,n_normal).clip(0),
        'nb_ope_reseau_autres': np.random.normal(10,5,n_normal).clip(0),
    })

    df_a = pd.DataFrame({
        'num_automate'      : [f'GAB_ANOM_{i:03d}' for i in np.random.randint(0,15,n_anom)],
        'annee'             : np.random.choice([2024,2025], n_anom),
        'mois'              : np.random.randint(1,13, n_anom),
        'type_gab_e_i'      : types_gab[n_normal:],
        'code_postal'       : codes_post[n_normal:],
        'longitude'         : np.random.uniform(-4.5,8.2, n_anom),
        'latitude'          : np.random.uniform(42.3,51.1, n_anom),
        'ret_nb'            : np.random.normal(1900,300,n_anom).clip(0),
        'ret_montant_moyen' : np.random.normal(420,80,n_anom).clip(100),
        'ret_montant_total' : np.random.normal(350000,60000,n_anom).clip(1000),
        'ret_montant_stddev': np.random.normal(180,40,n_anom).clip(5),
        'ret_nb_nuit'       : np.random.normal(280,60,n_anom).clip(0),
        'ret_nb_weekend'    : np.random.normal(520,80,n_anom).clip(0),
        'ret_pct_nuit'      : np.random.normal(35,8,n_anom).clip(0,100),
        'ret_pct_weekend'   : np.random.normal(46,8,n_anom).clip(0,100),
        'cap_nb'            : np.random.poisson(25,n_anom),
        'taux_capture_pct'  : np.random.normal(8.5,2.5,n_anom).clip(0),
        'nb_jours_actifs'   : np.random.randint(3,12,n_anom),
        'nb_ope_reseau_cb': np.random.normal(20,10,n_anom).clip(0),
        'nb_ope_reseau_visa': np.random.normal(180,40,n_anom).clip(0),
        'nb_ope_reseau_mastercard': np.random.normal(120,30,n_anom).clip(0),
        'nb_ope_reseau_interne': np.random.normal(80,20,n_anom).clip(0),
        'nb_ope_reseau_franfinance': np.random.normal(400,80,n_anom).clip(0),
        'nb_ope_reseau_accord': np.random.normal(25,10,n_anom).clip(0),
        'nb_ope_reseau_trionis': np.random.normal(15,8,n_anom).clip(0),
        'nb_ope_reseau_ppl': np.random.normal(12,6,n_anom).clip(0),
        'nb_ope_reseau_casino': np.random.normal(10,5,n_anom).clip(0),
        'nb_ope_reseau_configona': np.random.normal(8,4,n_anom).clip(0),
        'nb_ope_reseau_cos': np.random.normal(6,3,n_anom).clip(0),
        'nb_ope_reseau_jcb': np.random.normal(4,3,n_anom).clip(0),
        'nb_ope_reseau_postepargne': np.random.normal(5,3,n_anom).clip(0),
        'nb_ope_reseau_diners_et_discovery': np.random.normal(3,2,n_anom).clip(0),
        'nb_ope_reseau_autres': np.random.normal(10,5,n_anom).clip(0),
    })

    return pd.concat([df_n, df_a], ignore_index=True)

df = generer_dataset()

# Colonnes dérivées utiles
df['periode']          = df['annee'].astype(str) + '-' + df['mois'].astype(str).str.zfill(2)
df['pct_jours_actifs'] = df['nb_jours_actifs'] / 30 * 100
df['grande_ville']     = df['code_postal'].isin(['75001','69001','13001','33000','59000','06000'])
df['contexte_geo']     = np.where(df['grande_ville'], 'Grande ville', 'Zone intermédiaire / rurale')

# Réseau dominant
cols_reseau = ['nb_ope_reseau_franfinance', 'nb_ope_reseau_cb', 'nb_ope_reseau_trionis', 'nb_ope_reseau_ppl', 'nb_ope_reseau_mastercard', 'nb_ope_reseau_configona', 'nb_ope_reseau_interne', 'nb_ope_reseau_casino', 'nb_ope_reseau_accord', 'nb_ope_reseau_visa', 'nb_ope_reseau_cos', 'nb_ope_reseau_jcb', 'nb_ope_reseau_postepargne', 'nb_ope_reseau_diners_et_discovery', 'nb_ope_reseau_autres']
df['reseau_dominant']      = df[cols_reseau].idxmax(axis=1).str.replace('nb_ope_reseau_','')
df['nb_ope_total_reseau']  = df[cols_reseau].sum(axis=1)
df['concentration_reseau'] = df[cols_reseau].max(axis=1) / (df['nb_ope_total_reseau']+1)

print(f'📊 Dataset : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print(f'   → {df["num_automate"].nunique()} automates  |  {df["annee"].nunique()} années  |  {df["mois"].nunique()} mois distincts')
df.head(3)

## 3. 📊 Exploration des Données — Comportement Normal du Réseau

### Preuve 1 — Distribution des indicateurs clés

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16,9))
fig.suptitle('Distribution des indicateurs clés — Ensemble du réseau GAB', fontsize=15, fontweight='bold')

variables = [
    ('ret_nb',           'Nombre de retraits / mois',   COULEUR_NORMAL),
    ('ret_montant_moyen','Montant moyen retrait (€)',    COULEUR_NORMAL),
    ('taux_capture_pct', 'Taux de capture (%)',          COULEUR_ANOMALIE),
    ('ret_pct_nuit',     '% Retraits nocturnes',         COULEUR_ACCENT),
    ('ret_pct_weekend',  '% Retraits weekend',           COULEUR_ACCENT),
    ('concentration_reseau','Concentration réseau',      '#9C27B0'),
]

for ax, (col, titre, couleur) in zip(axes.flatten(), variables):
    data = df[col].dropna()
    ax.hist(data, bins=40, color=couleur, alpha=0.7, edgecolor='white')
    ax.axvline(data.mean(),   color='navy',  lw=2, ls='--', label=f'Moy: {data.mean():.1f}')
    ax.axvline(data.median(), color='green', lw=2, ls=':',  label=f'Med: {data.median():.1f}')
    ax.set_title(titre, fontweight='bold')
    ax.set_xlabel('')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 La majorité des GAB se concentre autour de valeurs stables → comportement NORMAL.')

### Preuve 2 — Impact du contexte géographique sur le volume de retraits

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Impact du contexte géographique — Grande ville vs Zone rurale',
             fontsize=13, fontweight='bold')

for ax, (col, titre) in zip(axes, [
    ('ret_nb',           'Nombre de retraits / mois'),
    ('taux_capture_pct', 'Taux de capture (%)'),
    ('ret_pct_nuit',     '% Retraits nocturnes'),
]):
    sns.boxplot(data=df, x='contexte_geo', y=col, ax=ax,
                palette={'Grande ville': COULEUR_NORMAL,
                         'Zone intermédiaire / rurale': COULEUR_ACCENT},
                width=0.5)
    ax.set_title(titre, fontweight='bold')
    ax.set_xlabel('')

plt.tight_layout()
plt.savefig('eda_geo_context.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Un GAB en grande ville avec 1200 retraits est NORMAL.')
print('   Un GAB en zone rurale avec 1200 retraits est ATYPIQUE.')
print('   → Le contexte géographique est indispensable pour interpréter le volume.')

### Preuve 3 — Répartition par réseau dominant

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Camembert réseau dominant
reseau_counts = df['reseau_dominant'].value_counts().head(8)
axes[0].pie(reseau_counts.values, labels=reseau_counts.index, autopct='%1.1f%%',
            colors=plt.cm.Set3(np.linspace(0,1,len(reseau_counts))))
axes[0].set_title('Répartition par réseau dominant', fontweight='bold')

# Boxplot taux capture par réseau dominant (top 6)
top_reseaux = df['reseau_dominant'].value_counts().head(6).index
df_top = df[df['reseau_dominant'].isin(top_reseaux)]
sns.boxplot(data=df_top, x='reseau_dominant', y='taux_capture_pct',
            ax=axes[1], palette='Set2')
axes[1].set_title('Taux de capture par réseau dominant', fontweight='bold')
axes[1].set_xlabel('Réseau dominant')
axes[1].set_ylabel('Taux capture (%)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_reseau.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. ⚙️ Feature Engineering — Variables enrichies pour le modèle

In [ ]:
# Variables dérivées pour le modèle
df['intensite_retrait']        = df['ret_nb'] / 30
df['ratio_capture']            = df['cap_nb'] / (df['ret_nb'] + 1)
df['cv_montant']               = df['ret_montant_stddev'] / (df['ret_montant_moyen'] + 1)
df['score_horaires_atypiques'] = (df['ret_pct_nuit'] / 10) + (df['ret_pct_weekend'] / 22)

# Z-scores contextualisés par type de GAB (Interne vs Externe)
# → On compare chaque GAB à ses HOMOLOGUES du même type, pas à tout le réseau
for type_gab in df['type_gab_e_i'].dropna().unique():
    mask = df['type_gab_e_i'] == type_gab
    for col in ['ret_nb','ret_montant_moyen','taux_capture_pct','ret_pct_nuit']:
        mu  = df.loc[mask, col].mean()
        sig = df.loc[mask, col].std()
        df.loc[mask, f'z_{col}_par_type'] = (df.loc[mask, col] - mu) / (sig + 1e-9)

print('✅ Features enrichies créées.')
print(f'   Colonnes z-score par type : {[c for c in df.columns if "z_" in c and "_par_type" in c]}')

## 5. 🔵 Clustering — Identification des Familles de Comportements

### Objectif : regrouper les GAB en familles homogènes AVANT de chercher les anomalies

Le clustering permet de comparer chaque GAB à ses **semblables** plutôt qu'à l'ensemble du réseau.
Un GAB rural sera comparé aux autres GAB ruraux — pas aux GAB de centre-ville.

In [ ]:
FEATURES_CLUSTER = [
    'ret_nb', 'ret_montant_moyen', 'taux_capture_pct',
    'ret_pct_nuit', 'ret_pct_weekend', 'concentration_reseau',
    'pct_jours_actifs', 'intensite_retrait', 'cv_montant'
]

X_clust = df[FEATURES_CLUSTER].fillna(0)
scaler_clust = StandardScaler()
X_clust_scaled = scaler_clust.fit_transform(X_clust)

# ── Méthode du coude pour choisir le nombre de clusters ──────────────────────
inertias, silhouettes = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust_scaled, labels, sample_size=2000))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'o-', color=COULEUR_NORMAL, lw=2)
axes[0].set_xlabel('Nombre de clusters K')
axes[0].set_ylabel('Inertie')
axes[0].set_title('Méthode du coude', fontweight='bold')

axes[1].plot(K_range, silhouettes, 'o-', color=COULEUR_ANOMALIE, lw=2)
axes[1].set_xlabel('Nombre de clusters K')
axes[1].set_ylabel('Score silhouette')
axes[1].set_title('Score silhouette', fontweight='bold')

plt.tight_layout()
plt.savefig('clustering_choix_k.png', dpi=150, bbox_inches='tight')
plt.show()

K_OPTIMAL = K_range[np.argmax(silhouettes)]
print(f'✅ Nombre de clusters optimal : K = {K_OPTIMAL} (meilleur score silhouette = {max(silhouettes):.3f})')

In [ ]:
# ── Entraînement KMeans final ─────────────────────────────────────────────────
kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_clust_scaled)

# ── Caractérisation des clusters ──────────────────────────────────────────────
cluster_stats = df.groupby('cluster')[FEATURES_CLUSTER + ['contexte_geo']].agg({
    'ret_nb'              : 'mean',
    'ret_montant_moyen'   : 'mean',
    'taux_capture_pct'    : 'mean',
    'ret_pct_nuit'        : 'mean',
    'ret_pct_weekend'     : 'mean',
    'pct_jours_actifs'    : 'mean',
    'concentration_reseau': 'mean',
    'contexte_geo'        : lambda x: x.value_counts().index[0],
}).round(2)

cluster_stats['nb_gab'] = df.groupby('cluster').size()

# ── Nommage automatique des familles ─────────────────────────────────────────
def nommer_cluster(row):
    if row['taux_capture_pct'] > 3.0:
        return '🔴 GAB atypiques — captures élevées'
    elif row['ret_nb'] > df['ret_nb'].quantile(0.75):
        return '🟢 GAB forte activité'
    elif row['ret_nb'] < df['ret_nb'].quantile(0.25):
        return '🔵 GAB faible activité'
    elif row['ret_pct_nuit'] > 20:
        return '🟠 GAB activité nocturne marquée'
    else:
        return '🟡 GAB activité moyenne'

cluster_stats['famille'] = cluster_stats.apply(nommer_cluster, axis=1)
df['famille'] = df['cluster'].map(cluster_stats['famille'])

print('📊 Familles de comportements identifiées :')
print(cluster_stats[['famille','nb_gab','ret_nb','taux_capture_pct','ret_pct_nuit','contexte_geo']])

### Preuve 4 — Visualisation des clusters (PCA 2D)

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_clust_scaled)
df['pca1'] = X_pca[:,0]
df['pca2'] = X_pca[:,1]

fig, ax = plt.subplots(figsize=(11, 7))
for i, (cluster_id, famille) in enumerate(cluster_stats['famille'].items()):
    mask = df['cluster'] == cluster_id
    ax.scatter(df[mask]['pca1'], df[mask]['pca2'],
               c=PALETTE_CLUSTERS[i % len(PALETTE_CLUSTERS)],
               label=f'Cluster {cluster_id} — {famille}',
               alpha=0.5, s=25, edgecolors='none')

ax.set_xlabel(f'PCA 1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PCA 2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('Familles de comportements GAB — Projection PCA', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9)
plt.tight_layout()
plt.savefig('clustering_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Chaque couleur = une famille de comportements. Les familles bien séparées = clustering pertinent.')

### Preuve 5 — Profil radar par famille

In [ ]:
DIMS_RADAR = ['ret_nb','ret_montant_moyen','taux_capture_pct',
              'ret_pct_nuit','ret_pct_weekend','concentration_reseau']
LABELS_RADAR = ['Nb retraits','Montant moyen','Taux capture',
                '% Nuit','% Weekend','Concentration réseau']

fig_radar = go.Figure()

for i, (cluster_id, famille) in enumerate(cluster_stats['famille'].items()):
    vals_norm = []
    for col in DIMS_RADAR:
        mn = df[col].quantile(0.05)
        mx = df[col].quantile(0.95)
        v  = cluster_stats.loc[cluster_id, col]
        vals_norm.append(max(0, min(1, (v - mn) / (mx - mn + 1e-9))))

    fig_radar.add_trace(go.Scatterpolar(
        r=vals_norm + [vals_norm[0]],
        theta=LABELS_RADAR + [LABELS_RADAR[0]],
        fill='toself',
        name=f'Cluster {cluster_id} — {famille}',
        line_color=PALETTE_CLUSTERS[i % len(PALETTE_CLUSTERS)],
        opacity=0.5
    ))

fig_radar.update_layout(
    polar={'radialaxis': {'visible': True, 'range': [0,1]}},
    title='Profil radar par famille de GAB',
    height=500
)
fig_radar.show()

## 6. 🤖 Détection d'Anomalies — Isolation Forest par Cluster

### Principe : on détecte les anomalies AU SEIN de chaque cluster

Un GAB est atypique s'il est différent de ses **homologues du même cluster**, pas de l'ensemble du réseau.

In [ ]:
FEATURES_IF = [
    'ret_nb', 'ret_montant_moyen', 'ret_montant_total',
    'ret_montant_stddev', 'taux_capture_pct', 'ratio_capture',
    'ret_pct_nuit', 'ret_pct_weekend', 'score_horaires_atypiques',
    'intensite_retrait', 'cv_montant', 'concentration_reseau',
    'pct_jours_actifs',
    'nb_ope_reseau_franfinance',
    'nb_ope_reseau_cb',
    'nb_ope_reseau_trionis',
    'nb_ope_reseau_ppl',
    'nb_ope_reseau_mastercard',
    'nb_ope_reseau_configona',
    'nb_ope_reseau_interne',
    'nb_ope_reseau_casino',
    'nb_ope_reseau_accord',
    'nb_ope_reseau_visa',
    'nb_ope_reseau_cos',
    'nb_ope_reseau_jcb',
    'nb_ope_reseau_postepargne',
    'nb_ope_reseau_diners_et_discovery',
    'nb_ope_reseau_autres',
]

FEATURES_IF = [c for c in FEATURES_IF if c in df.columns]

df['score_risque']  = 0.5
df['est_anomalie']  = 0
df['score_if_brut'] = 0.0

# ── Isolation Forest par cluster ───────────────────────────────────────────────
for cluster_id in df['cluster'].unique():
    mask = df['cluster'] == cluster_id
    X_sub = df.loc[mask, FEATURES_IF].fillna(0)

    scaler_if = StandardScaler()
    X_sub_scaled = scaler_if.fit_transform(X_sub)

    modele_if = IsolationForest(
        n_estimators=200,
        contamination=0.05,
        random_state=42,
        n_jobs=-1
    )
    preds  = modele_if.fit_predict(X_sub_scaled)
    scores = modele_if.score_samples(X_sub_scaled)

    # Normalisation [0,1]
    s_min, s_max = scores.min(), scores.max()
    scores_norm = 1 - (scores - s_min) / (s_max - s_min + 1e-9)

    df.loc[mask, 'score_risque']  = scores_norm
    df.loc[mask, 'est_anomalie']  = (preds == -1).astype(int)
    df.loc[mask, 'score_if_brut'] = scores

n_anom_total = df['est_anomalie'].sum()
print(f'🚨 GAB atypiques détectés : {n_anom_total} / {len(df)} ({n_anom_total/len(df)*100:.1f}%)')
print('\n📊 Répartition par cluster :')
print(df.groupby(['cluster','famille'])['est_anomalie'].agg(['sum','count']).rename(
    columns={'sum':'nb_atypiques','count':'total'}))

## 7. 📅 Consolidation Annuelle — Un GAB = Une Seule Ligne

### Problème résolu : dans v1, un même GAB apparaissait plusieurs fois (une fois par mois)
### Solution : on agrège par GAB sur l'année et on compte les mois atypiques

In [ ]:
# ── Table consolidée annuelle ──────────────────────────────────────────────────
df_annuel = df.groupby(['num_automate','annee']).agg(
    famille             = ('famille',            lambda x: x.mode()[0]),
    cluster             = ('cluster',            lambda x: x.mode()[0]),
    type_gab            = ('type_gab_e_i',       'first'),
    code_postal         = ('code_postal',        'first'),
    longitude           = ('longitude',          'first'),
    latitude            = ('latitude',           'first'),
    contexte_geo        = ('contexte_geo',        'first'),
    reseau_dominant     = ('reseau_dominant',     lambda x: x.mode()[0]),

    # Agrégats annuels
    ret_nb_annuel       = ('ret_nb',             'sum'),
    ret_nb_moy_mensuel  = ('ret_nb',             'mean'),
    montant_total_annuel= ('ret_montant_total',  'sum'),
    montant_moy_retrait = ('ret_montant_moyen',  'mean'),
    taux_capture_moy    = ('taux_capture_pct',   'mean'),
    taux_capture_max    = ('taux_capture_pct',   'max'),
    pct_nuit_moy        = ('ret_pct_nuit',       'mean'),
    pct_weekend_moy     = ('ret_pct_weekend',    'mean'),
    concentration_moy   = ('concentration_reseau','mean'),

    # Métriques atypicité
    nb_mois_total       = ('mois',              'count'),
    nb_mois_atypique    = ('est_anomalie',      'sum'),
    score_risque_moyen  = ('score_risque',      'mean'),
    score_risque_max    = ('score_risque',      'max'),
).reset_index()

# Ratio de mois atypiques sur l'année
df_annuel['pct_mois_atypique'] = (
    df_annuel['nb_mois_atypique'] / df_annuel['nb_mois_total'] * 100
).round(1)

# Seuil : GAB atypique sur l'année si atypique >= 2 mois
df_annuel['gab_atypique_annuel'] = (df_annuel['nb_mois_atypique'] >= 2).astype(int)

# Niveau d'alerte
df_annuel['niveau_alerte'] = pd.cut(
    df_annuel['nb_mois_atypique'],
    bins=[-1, 0, 2, 5, 12],
    labels=['✅ Normal', '🟡 Ponctuel (1-2 mois)', '🟠 Récurrent (3-5 mois)', '🔴 Persistant (6+ mois)']
)

n_atypiques_annuel = df_annuel['gab_atypique_annuel'].sum()
print(f'📊 Table consolidée annuelle : {len(df_annuel)} GAB uniques')
print(f'   → 🔴 GAB atypiques sur l\'année : {n_atypiques_annuel} ({n_atypiques_annuel/len(df_annuel)*100:.1f}%)')
print('\n📊 Répartition par niveau d\'alerte :')
print(df_annuel['niveau_alerte'].value_counts().sort_index())

## 8. 🔍 Justification Automatique du Caractère Atypique

### Pour chaque GAB atypique, le système génère automatiquement les raisons

In [ ]:
def generer_justification(row, df_ref):
    """
    Génère la liste des raisons qui rendent ce GAB atypique.
    Compare ses métriques à la médiane des GAB du même cluster et même type.
    """
    # Référence : GAB normaux du même cluster et type
    ref = df_ref[
        (df_ref['cluster'] == row['cluster']) &
        (df_ref['type_gab'] == row['type_gab']) &
        (df_ref['gab_atypique_annuel'] == 0)
    ]
    if len(ref) < 5:
        ref = df_ref[df_ref['gab_atypique_annuel'] == 0]

    raisons = []

    # Taux de capture
    ref_cap = ref['taux_capture_moy'].median()
    if ref_cap > 0 and row['taux_capture_moy'] / ref_cap > 2:
        raisons.append(
            f"⚠️  Taux de capture moyen : {row['taux_capture_moy']:.2f}% "
            f"(×{row['taux_capture_moy']/ref_cap:.1f} vs médiane cluster {ref_cap:.2f}%)"
        )

    # Volume retraits
    ref_ret = ref['ret_nb_moy_mensuel'].median()
    if ref_ret > 0:
        ratio_ret = row['ret_nb_moy_mensuel'] / ref_ret
        if ratio_ret > 2.0:
            raisons.append(
                f"📈 Volume retraits moyen : {row['ret_nb_moy_mensuel']:.0f}/mois "
                f"(×{ratio_ret:.1f} vs médiane cluster {ref_ret:.0f})"
            )
        elif ratio_ret < 0.3:
            raisons.append(
                f"📉 Volume retraits très faible : {row['ret_nb_moy_mensuel']:.0f}/mois "
                f"({ratio_ret*100:.0f}% de la médiane cluster {ref_ret:.0f})"
            )

    # Activité nocturne
    ref_nuit = ref['pct_nuit_moy'].median()
    if ref_nuit > 0 and row['pct_nuit_moy'] / ref_nuit > 2:
        raisons.append(
            f"🌙 Activité nocturne : {row['pct_nuit_moy']:.1f}% des retraits "
            f"(×{row['pct_nuit_moy']/ref_nuit:.1f} vs médiane cluster {ref_nuit:.1f}%)"
        )

    # Récurrence mensuelle
    if row['nb_mois_atypique'] >= 6:
        raisons.append(
            f"🔁 Récurrence persistante : atypique {row['nb_mois_atypique']} mois "
            f"sur {row['nb_mois_total']} ({row['pct_mois_atypique']:.0f}% de l'année)"
        )
    elif row['nb_mois_atypique'] >= 3:
        raisons.append(
            f"🔁 Récurrence notable : atypique {row['nb_mois_atypique']} mois "
            f"sur {row['nb_mois_total']}"
        )

    # Montant moyen
    ref_mnt = ref['montant_moy_retrait'].median()
    if ref_mnt > 0 and row['montant_moy_retrait'] / ref_mnt > 2:
        raisons.append(
            f"💰 Montant moyen : {row['montant_moy_retrait']:.0f}€ "
            f"(×{row['montant_moy_retrait']/ref_mnt:.1f} vs médiane {ref_mnt:.0f}€)"
        )

    # Concentration réseau
    if row['concentration_moy'] > 0.7:
        raisons.append(
            f"💳 Concentration réseau élevée : {row['concentration_moy']*100:.0f}% "
            f"sur le réseau '{row['reseau_dominant']}'"
        )

    if not raisons:
        raisons.append("ℹ️  Anomalie multivariée subtile — combinaison de signaux faibles")

    return ' | '.join(raisons)

# Application sur tous les GAB atypiques annuels
df_annuel['justification'] = df_annuel.apply(
    lambda row: generer_justification(row, df_annuel) if row['gab_atypique_annuel'] == 1 else '',
    axis=1
)

print(f'✅ Justifications générées pour {(df_annuel["gab_atypique_annuel"]==1).sum()} GAB atypiques.')
print('\n--- Exemple de justification ---')
exemple = df_annuel[df_annuel['gab_atypique_annuel']==1].nlargest(1,'score_risque_max')
print(f'GAB : {exemple.iloc[0]["num_automate"]}')
print(exemple.iloc[0]['justification'])

## 9. 📋 Top GAB Atypiques — Sans Doublons

### Chaque GAB n'apparaît qu'UNE SEULE FOIS, avec son niveau d'alerte et sa justification

In [ ]:
# ── Top 15 GAB atypiques annuels ──────────────────────────────────────────────
top_gab = df_annuel[df_annuel['gab_atypique_annuel']==1].nlargest(15, 'score_risque_max')[[
    'num_automate','annee','niveau_alerte','famille','type_gab',
    'code_postal','contexte_geo','reseau_dominant',
    'nb_mois_atypique','nb_mois_total','pct_mois_atypique',
    'score_risque_max','taux_capture_moy','ret_nb_moy_mensuel',
    'pct_nuit_moy','montant_moy_retrait','justification'
]].reset_index(drop=True)
top_gab.index += 1

print(f'🚨 TOP {len(top_gab)} GAB atypiques (un par GAB, sans doublons) :\n')
for _, row in top_gab.iterrows():
    print(f'{'='*70}')
    print(f'  {row["niveau_alerte"]} — {row["num_automate"]}  |  Score max : {row["score_risque_max"]:.3f}')
    print(f'  Famille : {row["famille"]}  |  Type : {row["type_gab"]}')
    print(f'  Localisation : {row["code_postal"]} ({row["contexte_geo"]})')
    print(f'  Atypique : {row["nb_mois_atypique"]} mois / {row["nb_mois_total"]}  ({row["pct_mois_atypique"]:.0f}%)')
    print(f'  Raisons :')
    for r in row['justification'].split(' | '):
        print(f'    {r}')

### Preuve 6 — Récurrence mensuelle des GAB atypiques

In [ ]:
# Histogramme : combien de mois chaque GAB atypique a-t-il été signalé ?
df_at = df_annuel[df_annuel['gab_atypique_annuel']==1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Récurrence du caractère atypique sur l\'année', fontsize=13, fontweight='bold')

# Histogramme nb mois atypiques
axes[0].hist(df_at['nb_mois_atypique'], bins=range(1,14),
             color=COULEUR_ANOMALIE, alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Nombre de mois atypiques dans l\'année')
axes[0].set_ylabel('Nombre de GAB')
axes[0].set_title('Distribution de la récurrence')
axes[0].axvline(6, color='black', ls='--', lw=1.5, label='Seuil persistant (6 mois)')
axes[0].legend()

# Barplot par niveau d'alerte
niveaux = df_annuel['niveau_alerte'].value_counts().sort_index()
couleurs_niveaux = ['#4CAF50','#FFC107','#FF9800','#F44336']
axes[1].barh(niveaux.index.astype(str), niveaux.values,
             color=couleurs_niveaux[:len(niveaux)], edgecolor='white')
axes[1].set_xlabel('Nombre de GAB')
axes[1].set_title('Répartition par niveau d\'alerte')
for i, v in enumerate(niveaux.values):
    axes[1].text(v+1, i, str(v), va='center', fontsize=10)

plt.tight_layout()
plt.savefig('recurrence_atypique.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. 🗺️ Carte Géographique — Familles et Anomalies

In [ ]:
fig_map = px.scatter_mapbox(
    df_annuel,
    lat='latitude', lon='longitude',
    color='niveau_alerte',
    color_discrete_map={
        '✅ Normal'                    : '#4CAF50',
        '🟡 Ponctuel (1-2 mois)'       : '#FFC107',
        '🟠 Récurrent (3-5 mois)'      : '#FF9800',
        '🔴 Persistant (6+ mois)'      : '#F44336',
    },
    size='score_risque_max',
    size_max=16,
    hover_name='num_automate',
    hover_data={
        'famille'          : True,
        'nb_mois_atypique' : True,
        'taux_capture_moy' : ':.2f',
        'score_risque_max' : ':.3f',
    },
    zoom=5,
    center={'lat': 46.8, 'lon': 2.3},
    mapbox_style='carto-positron',
    title='🗺️ Réseau GAB — Niveau d\'alerte annuel',
    height=580
)
fig_map.update_layout(margin={'r':0,'t':50,'l':0,'b':0})
fig_map.show()

## 11. 🌳 Arbre de Décision — Règles Métier Explicites

### Pourquoi ? → L'Isolation Forest détecte, l'arbre EXPLIQUE avec des règles lisibles

In [ ]:
if TREE_OK:
    FEATURES_TREE = [
        'ret_nb_moy_mensuel','montant_moy_retrait','taux_capture_moy',
        'pct_nuit_moy','pct_weekend_moy','concentration_moy','pct_jours_actifs'
    ]
    X_tree = df_annuel[FEATURES_TREE].fillna(0)
    y_tree = df_annuel['gab_atypique_annuel']

    arbre = DecisionTreeClassifier(max_depth=5, min_samples_leaf=10, random_state=42)
    arbre.fit(X_tree, y_tree)

    print('🌳 Règles apprises par l\'arbre de décision :')
    print('   (Ces règles expliquent directement POURQUOI un GAB est classé atypique)\n')
    print(export_text(arbre, feature_names=FEATURES_TREE, max_depth=4))

    # Importance des features
    importances = pd.Series(arbre.feature_importances_, index=FEATURES_TREE).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    importances.plot(kind='barh', ax=ax, color=COULEUR_NORMAL, edgecolor='white')
    ax.set_title('Importance des variables — Arbre de décision\n(directement interprétable)', fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig('arbre_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('⚠️  scikit-learn non disponible.')

## 12. 📌 Synthèse — Ce que l'Analyse a Révélé

In [ ]:
print('='*65)
print('  SYNTHÈSE — DÉTECTION COMPORTEMENTS ATYPIQUES GAB v2')
print('='*65)
print(f'  GAB analysés (table consolidée annuelle) : {len(df_annuel)}')
print(f'  Familles identifiées par clustering      : {K_OPTIMAL}')
print()
for niveau in df_annuel['niveau_alerte'].cat.categories:
    n = (df_annuel['niveau_alerte'] == niveau).sum()
    print(f'  {niveau} : {n} GAB ({n/len(df_annuel)*100:.1f}%)')
print()
print('  TOP 3 RAISONS D\'ATYPICITÉ (toutes familles) :')
raisons_counts = {}
for j in df_annuel[df_annuel['gab_atypique_annuel']==1]['justification']:
    for r in j.split(' | '):
        emoji = r[:2]
        raisons_counts[emoji] = raisons_counts.get(emoji, 0) + 1
for emoji, count in sorted(raisons_counts.items(), key=lambda x: -x[1])[:5]:
    label = {'⚠️':'Taux capture élevé','📈':'Volume retraits élevé',
             '🌙':'Activité nocturne','🔁':'Récurrence persistante',
             '💰':'Montant moyen élevé','💳':'Concentration réseau'}.get(emoji, emoji)
    print(f'    {emoji}  {label} : {count} GAB concernés')
print('='*65)
print()
print('  AMÉLIORATIONS v2 INTÉGRÉES :')
print('  ✅ Clustering KMeans — comparaison entre homologues')
print('  ✅ Contexte géographique — grande ville vs rural')
print('  ✅ Consolidation annuelle — un GAB = une ligne')
print('  ✅ Niveau d\'alerte — Normal / Ponctuel / Récurrent / Persistant')
print('  ✅ Justification automatique en français pour chaque GAB')
print('  ✅ Arbre de décision — règles métier explicites')
print('  ✅ Coordonnées GPS corrigées via BAN')
print('='*65)